# 05 — SedonaSpark GIS processing

SedonaSpark is included to learn distributed spatial data handling. It is not
a third headline benchmark result because its execution model and non-point
KNN behavior differ from SedonaDB's measured line-distance workflow.

**Learning objectives:** create a Sedona Spark session, inspect partitions,
read GeoParquet, register spatial SQL, and perform exact predicates.

In [1]:
import os
from pathlib import Path

from sedona.spark import SedonaContext

builder = (
    SedonaContext.builder()
    .master(os.environ.get("SPARK_MASTER", "local[*]"))
    .appName("sedona-israel-gis-learning")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
)
spark = SedonaContext.create(builder.getOrCreate())
print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/28 09:07:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.1


In [2]:
root = Path(os.environ.get("BENCHMARK_DATA_DIR", "/benchmark-data")) / "canonical"
roads_path = root / "israel_roads.parquet"
if not roads_path.exists():
    raise FileNotFoundError("Prepare canonical roads first")
roads = spark.read.format("geoparquet").load(str(roads_path))
roads.createOrReplaceTempView("roads")
print("Partitions:", roads.rdd.getNumPartitions())
spark.sql(
    """
    SELECT road_class, COUNT(*) AS fragments
    FROM roads
    WHERE is_general_driving
    GROUP BY road_class
    ORDER BY fragments DESC
    """
).show(truncate=False)

Partitions: 17


26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized reader. Falling back to parquet-mr
26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized reader. Falling back to parquet-mr
26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized reader. Falling back to parquet-mr
26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized reader. Falling back to parquet-mr
26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized reader. Falling back to parquet-mr
26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized reader. Falling back to parquet-mr
26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized reader. Falling back to parquet-mr
26/07/28 09:07:34 WARN GeoParquetFileFormat: GeoParquet currently does not support vectorized rea

+-------------+---------+
|road_class   |fragments|
+-------------+---------+
|residential  |134212   |
|tertiary     |65659    |
|secondary    |24620    |
|unclassified |13838    |
|trunk        |9894     |
|primary      |8873     |
|living_street|4909     |
|motorway     |2155     |
+-------------+---------+



## Partitioning is part of semantics and performance

Spark schedules partitions, not individual Python rows. Too few partitions
leave cores idle; too many create scheduler and shuffle overhead. Spatial
partitioning additionally affects candidate replication around boundaries.
Inspect physical plans rather than assuming a SQL function implies an index.

In [3]:
spark.sql(
    """
    SELECT road_id, road_class, ST_IsValid(geometry) AS valid
    FROM roads
    WHERE is_general_driving
    LIMIT 10
    """
).explain(mode="formatted")

== Physical Plan ==
CollectLimit (4)
+- Project (3)
   +- * Filter (2)
      +- Scan geoparquet  (1)


(1) Scan geoparquet 
Output [4]: [road_id#18, road_class#20, is_general_driving#22, geometry#24]
Batched: false
Location: InMemoryFileIndex [file:/benchmark-data/canonical/israel_roads.parquet]
PushedFilters: [IsNotNull(is_general_driving), EqualTo(is_general_driving,true)]
ReadSchema: struct<road_id:string,road_class:string,is_general_driving:boolean,geometry:binary>

(2) Filter [codegen id : 1]
Input [4]: [road_id#18, road_class#20, is_general_driving#22, geometry#24]
Condition : (isnotnull(is_general_driving#22) AND is_general_driving#22)

(3) Project
Output [3]: [road_id#18, road_class#20,  **org.apache.spark.sql.sedona_sql.expressions.ST_IsValid**   AS valid#72]
Input [4]: [road_id#18, road_class#20, is_general_driving#22, geometry#24]

(4) CollectLimit
Input [3]: [road_id#18, road_class#20, valid#72]
Arguments: 10




**Exercise:** compare `repartition(4)` and `repartition(28)` for a class
aggregation. That is a Spark scheduling exercise, not a replacement for the
controlled SedonaDB CPU-affinity benchmark.

Always stop the session before opening another kernel in the official
rootless local-mode image.

In [4]:
spark.stop()